# MODELLO PREVISIONALE PER IL MERCATO IMMOBILIARE

**RealEstateAI Solutions** punta a ottimizzare la valutazione dei prezzi delle proprietà immobiliari attraverso tecniche avanzate di regolarizzazione applicate alla regressione lineare. Questo notebook implementa e confronta tre approcci — **Ridge**, **Lasso** ed **Elastic Net** — analizzando quale offre le previsioni più accurate e il modello più interpretabile.

Il dataset contiene 545 immobili descritti da 13 variabili: caratteristiche fisiche (superficie, numero di stanze, piani), servizi (aria condizionata, riscaldamento, seminterrato) e posizionamento (zona di pregio, accesso alla strada principale). L'obiettivo è costruire un modello che — dato un insieme di caratteristiche — stimi il prezzo dell'immobile nel modo più accurato possibile.

## OBIETTIVO DEL PROGETTO

I modelli di regressione lineare classica (OLS) funzionano bene quando le feature sono poche e indipendenti, ma su dataset con variabili correlate tra loro tendono a produrre coefficienti instabili e gonfiati — un sintomo di overfitting. La regolarizzazione risolve il problema aggiungendo un termine di penalità alla funzione di costo, che "punisce" i coefficienti troppo grandi e li comprime verso lo zero.

**Le tre tecniche che confrontiamo:**

- **Ridge (L2)**: penalizza la somma dei *quadrati* dei coefficienti (`alpha * sum(beta²)`). Non azzera mai completamente un coefficiente, ma li riduce tutti. È particolarmente efficace quando ci sono molte feature debolmente correlate al target.

- **Lasso (L1)**: penalizza la somma dei *valori assoluti* dei coefficienti (`alpha * sum(|beta|)`). La geometria di questa penalità porta spesso i coefficienti meno rilevanti esattamente a zero, effettuando automaticamente una **selezione delle feature**.

- **Elastic Net (L1+L2)**: combina entrambe le penalità, bilanciate dal parametro `l1_ratio`. È più flessibile degli altri due, utile quando esistono gruppi di feature correlate (Lasso tenderebbe a selezionarne solo una per gruppo).

**Piano di lavoro:**
1. Caricare ed esplorare il dataset
2. Preprocessare i dati (encoding, normalizzazione)
3. Addestrare e ottimizzare ogni modello tramite Grid Search con cross-validation
4. Confrontare le performance con MSE, RMSE, MAE, R² e CV-MSE
5. Analizzare i coefficienti e i percorsi di regolarizzazione
6. Valutare la distribuzione dei residui

## SEZIONE 1: IMPORTAZIONI E CONFIGURAZIONE

In [ ]:
# ============================================================
# LIBRERIE UTILIZZATE E MOTIVAZIONE DELLE SCELTE
# ============================================================

import pandas as pd          # Gestione del dataset tabellare
import numpy as np           # Operazioni numeriche e array
import matplotlib.pyplot as plt  # Visualizzazioni di base
import seaborn as sns        # Visualizzazioni statistiche avanzate
import warnings
from scipy import stats      # Test statistici e Q-Q plot

# --- Modelli di regressione ---
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
# Ridge:           regolarizzazione L2 — comprime i coefficienti senza azzerarli
# Lasso:           regolarizzazione L1 — può portare coefficienti esattamente a zero
# ElasticNet:      combinazione L1+L2, bilanciata dal parametro l1_ratio
# LinearRegression: regressione OLS classica, usata come baseline di confronto

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler
# StandardScaler: normalizza le feature a media=0, deviazione standard=1.
# FONDAMENTALE per modelli con regolarizzazione: la penalità agisce sulla
# magnitudine dei coefficienti. Se 'area' è in m² (range 1650-16200) e
# 'bedrooms' va da 1 a 6, senza scaling 'area' avrebbe coefficienti molto più
# piccoli di 'bedrooms' anche se fosse più predittiva. La regolarizzazione la
# punirebbe ingiustamente. Con StandardScaler tutte le feature sono comparabili.

# --- Selezione del modello e validazione ---
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
# train_test_split: 80% training, 20% test
# GridSearchCV: ricerca sistematica dell'iperparametro alpha ottimale con CV
# cross_val_score: stima robusta delle performance tramite k-fold CV

# --- Metriche di valutazione ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# MSE (Mean Squared Error): media degli errori quadratici — penalizza gli errori grandi
# RMSE: radice del MSE — nella stessa unità monetaria del target, più leggibile
# MAE: media degli errori assoluti — meno sensibile agli outlier rispetto a MSE
# R²: proporzione di varianza del target spiegata dal modello (0=peggio della media, 1=perfetto)

warnings.filterwarnings('ignore')

# Configurazione grafica coerente per tutto il notebook
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Colori associati a ciascun modello: usati in tutti i grafici per coerenza visiva
COLORI = {
    'OLS':        '#78909C',   # grigio
    'Ridge':      '#1E88E5',   # blu
    'Lasso':      '#E53935',   # rosso
    'ElasticNet': '#43A047'    # verde
}

print('Librerie caricate correttamente.')

## SEZIONE 2: CARICAMENTO E PRIMA ESPLORAZIONE DEL DATASET

Il dataset è pubblicamente disponibile e viene caricato direttamente tramite URL — nessun file locale da gestire, il notebook è immediatamente eseguibile su Google Colab o qualsiasi altro ambiente Python. Prima di costruire qualsiasi modello, è essenziale capire con cosa stiamo lavorando: quanti dati abbiamo, che tipo di variabili contengono, e se ci sono problemi di qualità che potrebbero compromettere i risultati.

In [ ]:
# ============================================================
# CARICAMENTO DEL DATASET DA URL
# ============================================================
# Il dataset è liberamente accessibile online. pd.read_csv accetta URL HTTP/HTTPS
# direttamente, senza necessità di scaricare il file manualmente.
# In Google Colab questa chiamata funziona senza configurazioni aggiuntive.

DATASET_URL = 'https://proai-datasets.s3.eu-west-3.amazonaws.com/housing.csv'

df_raw = pd.read_csv(DATASET_URL)

print(f'Dataset caricato con successo!')
print(f'Dimensioni: {df_raw.shape[0]} righe x {df_raw.shape[1]} colonne')
print(f'\nVariabili presenti: {list(df_raw.columns)}')

# Normalizziamo i nomi delle colonne in minuscolo per coerenza
# (alcuni dataset hanno mix di maiuscole/minuscole)
df_raw.columns = df_raw.columns.str.lower().str.strip()
print(f'\nColonne dopo normalizzazione: {list(df_raw.columns)}')

In [ ]:
# Prima visualizzazione: le prime 10 righe per capire la struttura dei dati
# e verificare che il caricamento sia andato a buon fine
df_raw.head(10)

In [ ]:
# Panoramica dei tipi di dato e dei valori non-null per ogni colonna.
# Qui verifichiamo se il dataset contiene colonne testuali oppure se e' gia' numerico.
# Se tutte le colonne sono int/float, alcune variabili potrebbero comunque essere
# categoriche gia' codificate: lo controlliamo nella cella successiva.
df_raw.info()

In [ ]:
# ============================================================
# CONTROLLO VALORI UNICI: NUMERICHE REALI VS CATEGORICHE CODIFICATE
# ============================================================
# df_raw.info() mostra il tipo tecnico delle colonne, ma non basta sempre.
# Una colonna int64 con soli valori 0/1 spesso rappresenta una categoria binaria
# gia' trasformata in numero, non una misura quantitativa continua.

print('Valori distinti per colonna:\n')
for col in df_raw.columns:
    valori_unici = sorted(df_raw[col].dropna().unique())
    anteprima = valori_unici[:10]
    print(f'{col}: {df_raw[col].nunique()} valori distinti -> {anteprima}')

**Osservazione:**

Le colonne mainroad, guestroom, basement, hotwaterheating, airconditioning e prefarea hanno valori 0/1: sono variabili categoriche binarie gia\' codificate.

La colonna furnishingstatus ha valori 0/1/2: rappresenta tre categorie ordinate dello stato di arredamento.

In [ ]:
# ============================================================
# CONTROLLO QUALITÀ: VALORI MANCANTI E DUPLICATI
# ============================================================
print('--- VALORI MANCANTI PER COLONNA ---')
null_counts = df_raw.isnull().sum()
print(null_counts)
print(f'\nTotale valori mancanti: {null_counts.sum()}')

print('\n--- RIGHE DUPLICATE ---')
dup_count = df_raw.duplicated().sum()
print(f'Righe duplicate: {dup_count}')

if null_counts.sum() == 0 and dup_count == 0:
    print('\n✓ Nessun valore mancante e nessun duplicato: il dataset è già pulito.')

In [ ]:
# ============================================================
# DISTRIBUZIONE DELLE VARIABILI CATEGORICHE
# ============================================================
# Le colonne binarie (yes/no) e furnishingstatus sono categoriche.
# Contiamo le occorrenze per capire se ci sono classi sbilanciate
# e per identificare i valori da mappare durante l'encoding.

col_categoriche = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
                   'airconditioning', 'prefarea', 'furnishingstatus']

print('Distribuzione delle variabili categoriche:\n')
for col in col_categoriche:
    if col in df_raw.columns:
        print(f'[{col}]')
        print(df_raw[col].value_counts().to_string())
        print()

## SEZIONE 3: PREPROCESSING — ENCODING E PREPARAZIONE

Il dataset può contenere variabili categoriche in formato testuale oppure variabili categoriche già codificate come numeri. Nel secondo caso, come in una versione del dataset in cui tutte le colonne risultano `int64`, il tipo di dato da solo non basta per riconoscerle: bisogna guardare anche il significato della colonna e i valori distinti presenti.

**Strategia di encoding:**
- **Variabili binarie** (`mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea`): se sono testuali, mapping diretto `yes → 1`, `no → 0`; se sono già numeriche con valori `0/1`, non serve convertirle di nuovo. Sono comunque categoriche binarie, non misure continue.
- **`furnishingstatus`**: se è testuale, encoding ordinale `unfurnished → 0`, `semi-furnished → 1`, `furnished → 2`; se è già numerica con valori `0/1/2`, rappresenta già le tre categorie ordinate. Usiamo un valore ordinale perché c'è una progressione logica nel livello di arredo che si riflette plausibilmente anche sul prezzo: è ragionevole aspettarsi che un immobile completamente arredato valga più di uno parzialmente arredato, che a sua volta vale più di uno non arredato.

**Nota:** se il dataset ProAI fosse già pre-elaborato con valori numerici, la funzione di encoding lo rileva e salta la conversione.

In [ ]:
# ================================================================================
# FUNZIONE DI ENCODING DELLE VARIABILI CATEGORICHE -- Solo per test non necessaria
# ================================================================================

def encode_dataset(df):
    '''
    Converte le variabili categoriche in numeriche.
    Lavora su una copia del DataFrame per non alterare i dati originali.
    Rileva automaticamente se il dataset è già in formato numerico.
    '''
    df_enc = df.copy()

    # Verifica se ci sono colonne object da convertire
    colonne_object = df_enc.select_dtypes(include='object').columns.tolist()

    if len(colonne_object) == 0:
        print('Il dataset è già in formato numerico, nessun encoding necessario.')
        return df_enc

    print(f'Colonne da convertire: {colonne_object}')

    # Colonne binarie yes/no → 1/0
    colonne_binarie = ['mainroad', 'guestroom', 'basement',
                       'hotwaterheating', 'airconditioning', 'prefarea']
    for col in colonne_binarie:
        if col in df_enc.columns and df_enc[col].dtype == 'object':
            df_enc[col] = df_enc[col].map({'yes': 1, 'no': 0})

    # furnishingstatus: encoding ordinale (0=non arredato, 1=parziale, 2=completo)
    # L'ordine rispecchia la progressione nel valore dell'immobile
    if 'furnishingstatus' in df_enc.columns and df_enc['furnishingstatus'].dtype == 'object':
        mappa_arredo = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
        df_enc['furnishingstatus'] = df_enc['furnishingstatus'].map(mappa_arredo)

    print('\nEncoding completato.')
    return df_enc


df = encode_dataset(df_raw)
print('\nPrime righe dopo encoding:')
df.head(8)

## SEZIONE 4: ANALISI STATISTICA E VISUALIZZAZIONE ESPLORATIVA

Dopo avere analizzato con attenzione i dati numerici, possiamo ora analizzare le distribuzioni, cercare outlier e capire le relazioni tra variabili. Questa fase non è solo esplorazione: le informazioni che raccogliamo guidano le scelte di preprocessing e ci aiutano a interpretare i risultati dei modelli.

In [ ]:
# Statistiche descrittive: media, deviazione standard, quartili, min/max.
# Guardare questi numeri prima di costruire un modello è una buona abitudine:
# range molto diversi tra le feature confermano che lo scaling è necessario,
# e una std alta rispetto alla media può segnalare la presenza di outlier.
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df.describe().T

In [ ]:
# ============================================================
# DISTRIBUZIONE DEL TARGET: PRICE
# ============================================================
# Capire la distribuzione del target è fondamentale per la regressione lineare,
# che assume errori normalmente distribuiti. Una distribuzione asimmetrica
# del target può tradursi in residui non normali.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribuzione originale
axes[0].hist(df['price'], bins=30, color='#1E88E5', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribuzione del prezzo (originale)', fontsize=12)
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequenza')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--',
                label=f'Media: {df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df["price"].median():,.0f}')
axes[0].legend(fontsize=9)

# Distribuzione log-trasformata: comprime i prezzi molto alti
# e permette di confrontare la forma del target prima/dopo la trasformazione.
axes[1].hist(np.log1p(df['price']), bins=30, color='#43A047',
             edgecolor='white', alpha=0.85)
axes[1].set_title('Distribuzione di log(prezzo)', fontsize=12)
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequenza')

plt.suptitle('Distribuzione del Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

skewness = df['price'].skew()
print(f'Skewness del prezzo: {skewness:.3f}')
print(f'Una skewness positiva ({skewness:.2f}) indica coda lunga a destra:')
print(f'ci sono pochi immobili con prezzi molto alti che "tirano" la media.')
print(f"\nMediana: {df['price'].median():,.0f}")
print(f"Media:   {df['price'].mean():,.0f}")
print(f'Media > Mediana conferma l\'asimmetria positiva.')

**Osservazione sul grafico**

Il grafico di sinistra mostra la distribuzione originale della variabile `price`, cioè quanti immobili ricadono nelle diverse fasce di prezzo. Le due linee verticali aiutano a leggere il centro della distribuzione: la linea rossa indica la media, quella arancione la mediana. Il fatto che la media sia più alta della mediana conferma che ci sono alcuni immobili molto costosi che spostano il valore medio verso destra.

Il grafico di destra mostra invece gli stessi prezzi dopo una trasformazione logaritmica, cioè `log(Price + 1)`. Questa trasformazione comprime i valori molto grandi: le case più costose restano più costose, ma la loro distanza rispetto alle altre viene ridotta. Per questo la distribuzione appare più regolare e meno sbilanciata.

I due grafici sono stati messi a confronto per capire quanto il prezzo sia asimmetrico nella scala originale e quanto migliorerebbe la forma della distribuzione usando il logaritmo. È un controllo utile perché una regressione lineare lavora meglio quando il target e soprattutto i residui non sono troppo sbilanciati. In questo notebook il prezzo viene comunque mantenuto nella scala originale, così le metriche finali restano facili da leggere in termini economici.

In [ ]:
# ============================================================
# BOXPLOT DELLE FEATURE NUMERICHE CONTINUE
# ============================================================
# Il boxplot mostra: mediana (linea), IQR (box), baffi (1.5*IQR) e outlier (punti).
# È uno strumento rapido per identificare feature con range molto diversi
# (conferma la necessità dello scaling) e individuare valori anomali.

feature_continue = ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']

# grafico 1 | grafico 2 | grafico 3
# grafico 4 | grafico 5 | grafico 6
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(feature_continue):
    # mostro gli outlier come cherchi rossi di dimensione 4
    sns.boxplot(y=df[col], ax=axes[i], color='#90CAF9',
                flierprops=dict(marker='o', markerfacecolor='red',
                                markeredgecolor='red', markersize=4))
    # Imposta come titolo il nome della colonna.
    axes[i].set_title(f'{col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('')
    # Calcolo il primo quartile, cioè il valore sotto il quale si trova il 25% dei dati.
    q1 = df[col].quantile(0.25)
    # Calcolo il terzo quartile, cioè il valore sotto il quale si trova il 75% dei dati.
    q3 = df[col].quantile(0.75)
    # L’IQR misura l’ampiezza della parte centrale del dataset 
    # e contiene approsimativaente il 50% dei dati.
    iqr = q3 - q1
    # Seleziono le righe in cui il valore della feature è inferiore al limite
    # Q1 - 1.5 × IQR oppure superiore al limite Q3 + 1.5 × IQR.
    # Questi valori vengono considerati potenziali outlier secondo il metodo IQR. 
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
    if len(outliers) > 0:
        axes[i].set_xlabel(f'{len(outliers)} outlier (IQR method)', color='red', fontsize=9)

plt.suptitle('Boxplot delle feature numeriche', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Dettaglio outlier per valore:')
for col in feature_continue:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
    if len(outliers) > 0:
        print(f'\n[{col}] {len(outliers)} outlier totali')
        print(outliers[col].value_counts().sort_index().to_string())

**Osservazione sui boxplot**

Ogni boxplot riassume la distribuzione di una variabile numerica. La linea dentro il box rappresenta la mediana, cioè il valore centrale; il box contiene la parte più comune dei dati, tra primo e terzo quartile; i baffi mostrano l'intervallo considerato ancora normale; i punti fuori dai baffi sono possibili outlier.

Il confronto tra i grafici fa vedere subito che le variabili hanno scale molto diverse. `price` e `area` hanno valori molto più grandi e più dispersi, mentre `bedrooms`, `bathrooms`, `stories` e `parking` assumono pochi valori interi. Questo è importante perché Ridge, Lasso ed Elastic Net applicano una penalità sui coefficienti: senza normalizzazione, le variabili con scale diverse verrebbero trattate in modo poco equilibrato.

I punti estremi, soprattutto su `price` e `area`, non vengono eliminati automaticamente. Nel mercato immobiliare una casa molto grande o molto costosa può essere rara, ma non per forza sbagliata. Per questo il grafico serve più a prendere consapevolezza della forma dei dati che a decidere una rimozione meccanica degli outlier.

Una cosa importante: il numero scritto sotto ogni grafico indica quante righe del dataset sono considerate outlier, non quanti puntini si vedono a occhio. Per variabili discrete come `bedrooms`, `stories` e `parking`, più immobili possono avere esattamente lo stesso valore. Se, ad esempio, 41 immobili hanno `stories = 4`, nel boxplot quei 41 punti finiscono tutti sovrapposti nella stessa posizione e sembrano un solo punto rosso.

In [ ]:
# ============================================================
# RILEVAMENTO OUTLIER CON Z-SCORE
# ============================================================
# Lo Z-score misura quante deviazioni standard un valore dista dalla media.
# Soglia |z| > 3: in una distribuzione normale, solo lo 0.27% dei dati supera questa soglia.
# Valori oltre questa soglia meritano attenzione ma non vengono rimossi automaticamente:
# in un dataset immobiliare, una villa da 10M € è un outlier legittimo, non un errore.

from scipy.stats import zscore

print('--- OUTLIER RILEVATI CON Z-SCORE (|z| > 3) ---\n')

outlier_totali = set()
for col in feature_continue:
    z = np.abs(zscore(df[col].dropna()))
    n_outlier = (z > 3).sum()
    if n_outlier > 0:
        idx_outlier = df[col].dropna().index[z > 3]
        outlier_totali.update(idx_outlier)
        print(f'{col:<15}: {n_outlier} outlier')
    else:
        print(f'{col:<15}: nessun outlier')

print(f'\nRighe con almeno un outlier: {len(outlier_totali)} su {len(df)}')

**Decisione:** manteniamo tutti i campioni. Gli outlier in un dataset immobiliare spesso rappresentano proprietà reali (ville, case di lusso) e non errori di inserimento. Rimuoverli potrebbe ridurre la capacità del modello di gestire casi fuori dalla norma — che nella realtà esistono.

In [ ]:
# ============================================================
# MATRICE DI CORRELAZIONE
# ============================================================
# La correlazione di Pearson misura la relazione lineare tra coppie di variabili.
# Due aspetti ci interessano in particolare:
# 1. Correlazione con il target (price): indica le feature più predittive
# 2. Correlazione tra feature (multicollinearità): quando due predittori sono
#    molto correlati tra loro, la regressione OLS produce coefficienti instabili.
#    Ridge gestisce questo problema meglio degli altri metodi.

plt.figure(figsize=(13, 9))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            cbar=True, linewidths=0.4, linecolor='white',
            annot_kws={'size': 9})
plt.title('Matrice di Correlazione (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlazione con il target, ordinata per valore assoluto
print('\n--- Correlazione con il target "price" ---')
corr_target = corr_matrix['price'].drop('price').sort_values(key=abs, ascending=False)
for feature, val in corr_target.items():
    barra = '█' * int(abs(val) * 20)
    print(f'  {feature:<20}: {val:+.3f}  {barra}')

**Cosa ci dice la matrice di correlazione**

La relazione più evidente con il prezzo è quella di `area`: ha senso, perché la superficie è uno dei primi elementi che incidono sul valore di una casa. Anche `bathrooms`, `stories`, `airconditioning`, `parking` e `bedrooms` mostrano un legame interessante con il prezzo, quindi è plausibile che diano informazioni utili al modello.

Non emergono invece correlazioni estremamente alte tra le feature. Questo è positivo, perché riduce il rischio che due variabili raccontino quasi la stessa cosa e rendano instabili i coefficienti della regressione lineare. La regolarizzazione resta comunque utile, soprattutto per rendere il modello più stabile e meno sensibile ai singoli campioni.

## SEZIONE 5: PREPARAZIONE DEI DATI

Prima di addestrare qualsiasi modello dobbiamo:
1. Separare feature (X) e target (y)
2. Dividere in training set (80%) e test set (20%)
3. Normalizzare le feature con StandardScaler

La normalizzazione è il passo più delicato. Come già spiegato nell'introduzione, senza di essa la regolarizzazione penalizzerebbe ingiustamente le feature con valori assoluti più piccoli. **ATTENZIONE**: lo scaler deve essere addestrato SOLO sul training set. Applicarlo sull'intero dataset prima della divisione causerebbe *data leakage* — il modello "vedrebbe" le statistiche del test set durante la normalizzazione, ottimismo che non si ripete nel mondo reale.

In [ ]:
# ============================================================
# SEPARAZIONE FEATURE / TARGET
# ============================================================

# Target: il prezzo dell'immobile (variabile da prevedere)
y = df['price']

# Feature: tutte le altre colonne
X = df.drop('price', axis=1)

FEATURE_NAMES = X.columns.tolist()

print(f'Feature utilizzate ({len(FEATURE_NAMES)}):')
for f in FEATURE_NAMES:
    print(f'  - {f}')
print(f'\nTarget: price')
print(f'  Range: {y.min():,.0f} — {y.max():,.0f}')
print(f'  Media: {y.mean():,.0f}')

# ============================================================
# SUDDIVISIONE TRAIN/TEST  80% / 20%
# ============================================================
# test_size=0.2: 80% per addestrare, 20% per la valutazione finale.
# Il 20% corrisponde a ~109 campioni: abbastanza per una stima affidabile.
# random_state=42: seed fisso per riproducibilità esatta dei risultati.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTraining set: {X_train.shape[0]} campioni ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:     {X_test.shape[0]} campioni ({X_test.shape[0]/len(X)*100:.0f}%)')

# ============================================================
# NORMALIZZAZIONE — fit SOLO su X_train
# ============================================================
# fit_transform: calcola media e std dal training set e le applica
# transform: applica le STESSE statistiche al test set (no data leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Convertiamo in DataFrame per mantenere i nomi delle colonne
# (utile per l'analisi dei coefficienti)
X_train_s = pd.DataFrame(X_train_s, columns=FEATURE_NAMES)
X_test_s  = pd.DataFrame(X_test_s,  columns=FEATURE_NAMES)

print(f'\nMedia training set scalato (deve essere ≈ 0 per ogni colonna):')
print(X_train_s.mean().round(6).to_string())
print(f'\nStd training set scalato (deve essere ≈ 1 per ogni colonna):')
print(X_train_s.std().round(6).to_string())

In [ ]:
# ============================================================
# FUNZIONI DI UTILITÀ — VALUTAZIONE E ANALISI DEI MODELLI
# ============================================================

def evaluate_regression_model(nome, modello, X_train, X_test, y_train, y_test, cv=5):
    '''
    Addestra e valuta un modello di regressione sul test set e in cross-validation.

    Restituisce un dizionario con tutte le metriche:
    - MSE, RMSE, MAE: misurano l'entità dell'errore (più basso = meglio)
    - R²: varianza spiegata (più alto = meglio, max 1.0)
    - CV_MSE: MSE medio sulla cross-validation sul training set
      (stima più robusta della generalizzazione rispetto al solo test set)
    - y_pred: array delle previsioni sul test set (per analisi residui)
    '''
    # 1. Addestramento sul training set completo
    modello.fit(X_train, y_train)

    # 2. Previsioni sul test set
    y_pred = modello.predict(X_test)

    # 3. Metriche sul test set
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    # 4. Cross-validation sul training set
    # neg_mean_squared_error: sklearn massimizza gli score, quindi neghiamo MSE
    # cross_val_score clona internamente il modello — non altera lo stato dell'oggetto
    cv_scores = cross_val_score(modello, X_train, y_train,
                                cv=cv, scoring='neg_mean_squared_error')
    cv_mse = -cv_scores.mean()
    cv_std =  cv_scores.std()

    return {
        'Modello':         nome,
        'MSE':             mse,
        'RMSE':            rmse,
        'MAE':             mae,
        'R²':              r2,
        'CV-MSE (media)':  cv_mse,
        'CV-MSE (std)':    cv_std,
        'y_pred':          y_pred
    }


def conta_coeff_non_nulli(modello, soglia=1e-10):
    '''
    Conta i coefficienti con valore assoluto > soglia.
    Lasso e ElasticNet possono portare coefficienti esattamente a zero:
    questo è il meccanismo di feature selection automatica.
    '''
    return int(np.sum(np.abs(modello.coef_) > soglia))


print('Funzioni di utilità caricate.')

## SEZIONE 6: MODELLO BASELINE — REGRESSIONE LINEARE OLS

Prima di applicare la regolarizzazione, addestriamo una regressione lineare classica (Ordinary Least Squares). Questo modello minimizza la somma degli errori quadratici senza alcuna penalità sui coefficienti. Serve come **punto di riferimento**: se i modelli regolarizzati non migliorano rispetto a questo, la regolarizzazione non sta aiutando — o il dataset è già abbastanza semplice da non soffrire di overfitting.

In [ ]:
# ============================================================
# BASELINE: REGRESSIONE LINEARE SENZA REGOLARIZZAZIONE
# ============================================================

lr_baseline = LinearRegression()
risultati_ols = evaluate_regression_model(
    'OLS', lr_baseline,
    X_train_s, X_test_s, y_train, y_test
)

print('=== BASELINE: Regressione Lineare OLS ===')
print(f"MSE:             {risultati_ols['MSE']:>18,.2f}")
print(f"RMSE:            {risultati_ols['RMSE']:>18,.2f}  ← errore medio in unità monetarie")
print(f"MAE:             {risultati_ols['MAE']:>18,.2f}")
print(f"R²:              {risultati_ols['R²']:>18.4f}")
print(f"CV-MSE (media):  {risultati_ols['CV-MSE (media)']:>18,.2f}")
print(f"CV-MSE (std):    {risultati_ols['CV-MSE (std)']:>18,.2f}")
print(f'Coeff. non nulli: {conta_coeff_non_nulli(lr_baseline)}/{len(FEATURE_NAMES)} (OLS non azzera mai)')

# Coefficienti del baseline (ordinati per importanza assoluta)
df_coeff_ols = pd.DataFrame({
    'Feature':    FEATURE_NAMES,
    'Coeff. OLS': lr_baseline.coef_
}).set_index('Feature').sort_values('Coeff. OLS', key=abs, ascending=False)

print('\nCoefficiente per feature (dati scalati):')
print(df_coeff_ols.round(0).to_string())

## SEZIONE 7: RIDGE REGRESSION (Regolarizzazione L2)

Ridge aggiunge alla funzione di costo OLS un termine proporzionale alla **somma dei quadrati** dei coefficienti:

```
Costo Ridge = Σ(yᵢ - ŷᵢ)² + α · Σ(βⱼ²)
```

L'effetto geometrico è che Ridge preferisce coefficienti piccoli e distribuiti uniformemente tra le feature. Al contrario dell'OLS, i coefficienti non diventano mai esattamente zero — tutte le feature restano nel modello, ma compresse. Questa proprietà lo rende robusto in presenza di multicollinearità: quando due feature sono correlate, OLS assegna coefficienti instabili e grandi; Ridge li divide in modo equilibrato tra entrambe.

L'iperparametro **α** controlla la forza della regolarizzazione: α=0 è equivalente a OLS, α molto grande porta tutti i coefficienti verso zero. Il valore ottimale va trovato empiricamente.

In [ ]:
# ============================================================
# RIDGE — RICERCA DELL'ALPHA OTTIMALE CON GRID SEARCH
# ============================================================
# Usiamo una scala logaritmica per l'alpha: copre ordini di grandezza
# diversi con pochi punti. Da 0.01 (quasi OLS) a 100000 (regolarizzazione forte).
# 5-fold cross-validation: standard per dataset di queste dimensioni.

alphas_ridge = np.logspace(-2, 5, 100)

ridge_gs = GridSearchCV(
    Ridge(max_iter=10000),
    param_grid={'alpha': alphas_ridge},
    cv=5,
    scoring='neg_mean_squared_error',
    return_train_score=True
)
ridge_gs.fit(X_train_s, y_train)

ALPHA_RIDGE = ridge_gs.best_params_['alpha']
print(f'Alpha ottimale per Ridge: {ALPHA_RIDGE:.4f}')
print(f'CV-MSE migliore:          {-ridge_gs.best_score_:,.2f}')

# Grafico: CV-MSE al variare di alpha
cv_means = -ridge_gs.cv_results_['mean_test_score']
cv_stds  =  ridge_gs.cv_results_['std_test_score']

plt.figure(figsize=(10, 4))
plt.plot(alphas_ridge, cv_means, color=COLORI['Ridge'], linewidth=2, label='CV-MSE medio')
plt.fill_between(alphas_ridge,
                 cv_means - cv_stds,
                 cv_means + cv_stds,
                 alpha=0.2, color=COLORI['Ridge'], label='±1 std')
plt.axvline(x=ALPHA_RIDGE, color='black', linestyle='--', linewidth=1.5,
            label=f'Alpha ottimale = {ALPHA_RIDGE:.2f}')
plt.xscale('log')
plt.xlabel('Alpha (scala logaritmica)')
plt.ylabel('CV-MSE')
plt.title('Ridge: CV-MSE al variare di Alpha', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

**Osservazione sul grafico Ridge**

Questo grafico mostra come cambia l'errore medio in cross-validation al variare di `alpha`, cioè della forza della regolarizzazione Ridge. I coefficienti sono i pesi che il modello assegna alle variabili: ad esempio quanto conta `area`, quanto conta `bathrooms`, quanto conta `airconditioning` e così via nella previsione del prezzo.

A sinistra del grafico `alpha` è molto basso, quindi la penalità è quasi assente: il modello è libero di assegnare pesi anche grandi alle variabili, come farebbe una regressione lineare classica. Andando verso destra `alpha` aumenta: Ridge inizia a limitare questi pesi, cioè spinge il modello a usare coefficienti più piccoli e meno estremi.

Questo però non significa che andare verso destra migliori sempre il risultato. Nel grafico, infatti, dopo un certo punto il CV-MSE cresce: vuol dire che la penalità è diventata troppo forte, i coefficienti sono stati ridotti troppo e il modello non riesce più a spiegare bene il prezzo.

La linea nera verticale indica il valore di `alpha` scelto dalla Grid Search. Il punto migliore è quello in cui il CV-MSE è più basso: significa che, mediamente sui fold di validazione, il modello commette l'errore minore. La fascia colorata intorno alla curva rappresenta la variabilità tra i fold: se è stretta, il risultato è più stabile; se è larga, il modello dipende di più da come vengono suddivisi i dati.

Il senso del grafico è quindi scegliere una regolarizzazione abbastanza forte da stabilizzare i coefficienti, ma non così forte da rendere il modello troppo rigido.

In [ ]:
# ============================================================
# RIDGE — ADDESTRAMENTO E VALUTAZIONE FINALE
# ============================================================

ridge_finale = Ridge(alpha=ALPHA_RIDGE, max_iter=10000)
risultati_ridge = evaluate_regression_model(
    'Ridge', ridge_finale,
    X_train_s, X_test_s, y_train, y_test
)

print(f'=== RIDGE REGRESSION (alpha = {ALPHA_RIDGE:.4f}) ===')
print(f"MSE:             {risultati_ridge['MSE']:>18,.2f}")
print(f"RMSE:            {risultati_ridge['RMSE']:>18,.2f}")
print(f"MAE:             {risultati_ridge['MAE']:>18,.2f}")
print(f"R²:              {risultati_ridge['R²']:>18.4f}")
print(f"CV-MSE (media):  {risultati_ridge['CV-MSE (media)']:>18,.2f}")
print(f"CV-MSE (std):    {risultati_ridge['CV-MSE (std)']:>18,.2f}")
print(f'Coeff. non nulli: {conta_coeff_non_nulli(ridge_finale)}/{len(FEATURE_NAMES)}')
print(f'\n(Ridge non azzera mai i coefficienti: tutti {len(FEATURE_NAMES)} predittori restano attivi)')

**Considerazioni su Ridge**

Con Ridge il valore migliore trovato dalla Grid Search è `alpha ≈ 29.15`. Nel grafico questo valore cade nella zona in cui il CV-MSE è ancora basso e stabile, prima della salita evidente che si vede nella parte destra. Questo è un buon segnale: la regolarizzazione sta correggendo il modello, ma non è così forte da renderlo troppo rigido.

Il risultato finale sul test set è abbastanza chiaro: `RMSE ≈ 1,330,562`, `MAE ≈ 957,661` e `R² ≈ 0.6497`. In pratica, l'errore medio assoluto è intorno a 958 mila unità monetarie, mentre l'R² indica che il modello riesce a spiegare circa il 65% della variabilità del prezzo. Non è un modello perfetto, ma cattura una parte significativa della relazione tra caratteristiche dell'immobile e prezzo.

Conclusione: Ridge in questo progetto è una versione più stabile della regressione lineare classica. Non serve a semplificare il modello, ma a renderlo meno sensibile ai dati e più regolare nei coefficienti.

## SEZIONE 8: LASSO REGRESSION (Regolarizzazione L1)

Lasso (Least Absolute Shrinkage and Selection Operator) usa la **norma L1** come penalità:

```
Costo Lasso = Σ(yᵢ - ŷᵢ)² + α · Σ|βⱼ|
```

La differenza fondamentale rispetto a Ridge sta nella geometria della penalità. La norma L1 ha degli "angoli" alle intersezioni con gli assi — è in questi punti che la soluzione ottimale tende a cadere, portando alcuni coefficienti esattamente a zero. In pratica, Lasso fa **feature selection automatica**: le feature meno rilevanti vengono eliminate, rendendo il modello più semplice e interpretabile.

Quando preferire Lasso: quando si sospetta che solo alcune delle variabili disponibili siano effettivamente predittive, e si vuole identificare quali.

In [ ]:
# ============================================================
# LASSO — RICERCA DELL'ALPHA OTTIMALE
# ============================================================
# Lasso tende a richiedere alpha più piccoli rispetto a Ridge
# per un effetto di regolarizzazione comparabile, perché la penalità
# L1 è più aggressiva di L2 nell'azzerare i coefficienti.
# Partiamo da 1e-4 per esplorare anche valori molto piccoli (quasi OLS).

alphas_lasso = np.logspace(-4, 3, 100)

lasso_gs = GridSearchCV(
    Lasso(max_iter=20000),
    param_grid={'alpha': alphas_lasso},
    cv=5,
    scoring='neg_mean_squared_error',
    return_train_score=True
)
lasso_gs.fit(X_train_s, y_train)

ALPHA_LASSO = lasso_gs.best_params_['alpha']
print(f'Alpha ottimale per Lasso: {ALPHA_LASSO:.6f}')
print(f'CV-MSE migliore:          {-lasso_gs.best_score_:,.2f}')

# Grafico CV-MSE vs alpha
cv_means_l = -lasso_gs.cv_results_['mean_test_score']
cv_stds_l  =  lasso_gs.cv_results_['std_test_score']

plt.figure(figsize=(10, 4))
plt.plot(alphas_lasso, cv_means_l, color=COLORI['Lasso'], linewidth=2, label='CV-MSE medio')
plt.fill_between(alphas_lasso,
                 cv_means_l - cv_stds_l,
                 cv_means_l + cv_stds_l,
                 alpha=0.2, color=COLORI['Lasso'], label='±1 std')
plt.axvline(x=ALPHA_LASSO, color='black', linestyle='--', linewidth=1.5,
            label=f'Alpha ottimale = {ALPHA_LASSO:.4f}')
plt.xscale('log')
plt.xlabel('Alpha (scala logaritmica)')
plt.ylabel('CV-MSE')
plt.title('Lasso: CV-MSE al variare di Alpha', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

**Osservazione sul grafico Lasso**

Il grafico mette in relazione `alpha` e CV-MSE. Nel caso di Lasso, `alpha` non controlla solo quanto vengono ridotti i coefficienti: può anche portarli esattamente a zero, quindi può eliminare alcune variabili dal modello.

In questo risultato la curva del CV-MSE è quasi piatta: l'errore cambia pochissimo tra valori di `alpha` molto piccoli e valori più grandi. Questo significa che, per questo dataset, Lasso non mostra un miglioramento netto dell'errore al variare della penalità. Il modello sembra abbastanza stabile, ma non c'è un punto della curva che si distingua chiaramente come molto migliore degli altri.

La linea verticale è sull'estremo destro del grafico perché la Grid Search ha scelto `alpha = 1000`, cioè il valore più alto tra quelli provati. Questo va letto con attenzione: non significa automaticamente che `1000` sia il miglior valore possibile in assoluto, ma che è il migliore dentro la griglia testata. Quando il valore ottimale cade su un bordo della griglia, può avere senso provare anche valori più alti per verificare se l'errore continua a migliorare o se inizia a peggiorare.

Il risultato principale, quindi, è questo: Lasso riesce a mantenere un errore simile agli altri valori di `alpha`, ma bisogna controllare quanti coefficienti ha azzerato. Se semplifica il modello senza peggiorare molto il CV-MSE, può essere utile per avere un modello più interpretabile.

In [ ]:
# ============================================================
# LASSO — TEST AGGIUNTIVO CON ALPHA PIU' ALTI
# ============================================================
# Nel grafico precedente il valore migliore era sul bordo destro della griglia.
# Questo test verifica se, provando alpha piu' alti, il CV-MSE migliora ancora
# oppure se la penalita' diventa eccessiva e il modello inizia a peggiorare.

ALPHA_LASSO_BASE = ALPHA_LASSO
CV_MSE_LASSO_BASE = -lasso_gs.best_score_

alphas_lasso_extended = np.logspace(3, 6, 80)  # da 1.000 a 1.000.000

lasso_gs_extended = GridSearchCV(
    Lasso(max_iter=50000),
    param_grid={'alpha': alphas_lasso_extended},
    cv=5,
    scoring='neg_mean_squared_error',
    return_train_score=True
)
lasso_gs_extended.fit(X_train_s, y_train)

ALPHA_LASSO_EXT = lasso_gs_extended.best_params_['alpha']
CV_MSE_LASSO_EXT = -lasso_gs_extended.best_score_

print('--- Confronto ricerca Lasso ---')
print(f'Miglior alpha griglia iniziale:  {ALPHA_LASSO_BASE:,.6f}')
print(f'CV-MSE griglia iniziale:         {CV_MSE_LASSO_BASE:,.2f}')
print(f'Miglior alpha griglia estesa:    {ALPHA_LASSO_EXT:,.6f}')
print(f'CV-MSE griglia estesa:           {CV_MSE_LASSO_EXT:,.2f}')

if CV_MSE_LASSO_EXT < CV_MSE_LASSO_BASE:
    ALPHA_LASSO = ALPHA_LASSO_EXT
    print('\nRisultato: la griglia estesa migliora il CV-MSE.')
    print(f'Usiamo quindi alpha = {ALPHA_LASSO:,.6f} per il modello Lasso finale.')
else:
    ALPHA_LASSO = ALPHA_LASSO_BASE
    print('\nRisultato: la griglia estesa non migliora il CV-MSE.')
    print(f'Manteniamo quindi alpha = {ALPHA_LASSO:,.6f} per il modello Lasso finale.')

# Grafico del solo intervallo esteso
cv_means_l_ext = -lasso_gs_extended.cv_results_['mean_test_score']
cv_stds_l_ext = lasso_gs_extended.cv_results_['std_test_score']

plt.figure(figsize=(10, 4))
plt.plot(alphas_lasso_extended, cv_means_l_ext, color=COLORI['Lasso'],
         linewidth=2, label='CV-MSE medio - griglia estesa')
plt.fill_between(alphas_lasso_extended,
                 cv_means_l_ext - cv_stds_l_ext,
                 cv_means_l_ext + cv_stds_l_ext,
                 alpha=0.2, color=COLORI['Lasso'], label='±1 std')
plt.axvline(x=ALPHA_LASSO_EXT, color='black', linestyle='--', linewidth=1.5,
            label=f'Miglior alpha esteso = {ALPHA_LASSO_EXT:.2f}')
plt.axhline(y=CV_MSE_LASSO_BASE, color='gray', linestyle=':', linewidth=1.5,
            label='CV-MSE migliore griglia iniziale')
plt.xscale('log')
plt.xlabel('Alpha (scala logaritmica)')
plt.ylabel('CV-MSE')
plt.title('Lasso: test con valori di alpha piu\' alti', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

**Osservazione sul test esteso di Lasso**

Il test esteso conferma che aveva senso controllare valori di `alpha` più alti: la griglia iniziale si fermava a `1000`, ma il valore migliore trovato dopo l'estensione è circa `2616.50`. Il CV-MSE migliora leggermente, passando da circa `1,075,007,729,540.58` a circa `1,074,989,132,439.39`.

Il miglioramento però è molto piccolo rispetto alla scala dell'errore. Questo significa che Lasso non sta cambiando radicalmente le prestazioni: sta solo trovando un punto un po' più preciso nella zona in cui la curva è quasi piatta. La scelta di aggiornare `ALPHA_LASSO` è corretta, ma non va letta come un salto importante di qualità del modello.

La parte destra del grafico è invece molto chiara: quando `alpha` diventa troppo alto, il CV-MSE cresce rapidamente. In quel punto la penalità è eccessiva, troppi coefficienti vengono ridotti o azzerati, e il modello perde informazioni utili per prevedere il prezzo.

Conclusione: usiamo `alpha ≈ 2616.50` per il Lasso finale, ma il risultato conferma soprattutto che bisogna evitare valori molto più alti, perché peggiorano nettamente la capacità predittiva.

In [ ]:
# ============================================================
# LASSO — ADDESTRAMENTO E VALUTAZIONE FINALE
# ============================================================
# Usiamo ALPHA_LASSO dopo il test esteso.
# Se la griglia estesa ha migliorato il CV-MSE, ALPHA_LASSO contiene il nuovo valore;
# altrimenti contiene ancora il miglior valore della griglia iniziale.

lasso_finale = Lasso(alpha=ALPHA_LASSO, max_iter=50000)
risultati_lasso = evaluate_regression_model(
    'Lasso', lasso_finale,
    X_train_s, X_test_s, y_train, y_test
)

n_non_zero_l = conta_coeff_non_nulli(lasso_finale)
n_zero_l     = len(FEATURE_NAMES) - n_non_zero_l

print(f'Alpha Lasso usato per la valutazione finale: {ALPHA_LASSO:,.6f}')
print(f'=== LASSO REGRESSION (alpha = {ALPHA_LASSO:.6f}) ===')
print(f"MSE:             {risultati_lasso['MSE']:>18,.2f}")
print(f"RMSE:            {risultati_lasso['RMSE']:>18,.2f}")
print(f"MAE:             {risultati_lasso['MAE']:>18,.2f}")
print(f"R²:              {risultati_lasso['R²']:>18.4f}")
print(f"CV-MSE (media):  {risultati_lasso['CV-MSE (media)']:>18,.2f}")
print(f"CV-MSE (std):    {risultati_lasso['CV-MSE (std)']:>18,.2f}")
print(f'Coeff. non nulli: {n_non_zero_l}/{len(FEATURE_NAMES)}')
print(f'Coeff. azzerati:  {n_zero_l}/{len(FEATURE_NAMES)}  ← feature eliminate automaticamente')

# Dettaglio coefficienti: chi è stato azzerato?
print('\nCoefficiente per feature (Lasso):')
df_coeff_lasso = pd.DataFrame({
    'Coeff. Lasso': lasso_finale.coef_,
    'Azzerato?':    ['SI' if abs(c) <= 1e-10 else '  ' for c in lasso_finale.coef_]
}, index=FEATURE_NAMES).sort_values('Coeff. Lasso', key=abs, ascending=False)
print(df_coeff_lasso.to_string())

**Considerazioni su Lasso**

In questo risultato, lasso non seleziona automaticamente le variabili: con `alpha ≈ 2616.50` il modello mantiene attivi tutti i 12 predittori (`Coeff. azzerati: 0/12`). Quindi, su questo dataset, Lasso non sta semplificando davvero il modello rispetto a Ridge o OLS.

Vediamo che la penalità L1 riduce i pesi, ma non abbastanza da eliminare variabili senza peggiorare la previsione. Questo suggerisce che anche le feature con peso più piccolo contengono ancora un po' di informazione utile per stimare il prezzo.

Guardando i coefficienti ordinati, le variabili con peso maggiore sono `bathrooms`, `area`, `airconditioning`, `stories` e `prefarea`. È una lettura coerente con il problema immobiliare: superficie, numero di bagni, comfort e posizione tendono ad avere un impatto importante sul prezzo. Variabili come `bedrooms` e `guestroom` restano nel modello, ma con un peso più contenuto.

Conclusione: in questo caso Lasso non porta un vero vantaggio di parsimonia, perché non elimina feature. Rimane comunque utile come confronto, perché mostra che una penalità più aggressiva non riesce a semplificare il modello senza rischiare di perdere informazione.

## SEZIONE 9: ELASTIC NET REGRESSION (L1 + L2)

Elastic Net combina entrambe le penalità, bilanciate dal parametro `l1_ratio` (ρ):

```
Costo EN = Σ(yᵢ - ŷᵢ)² + α · [ρ · Σ|βⱼ| + (1-ρ)/2 · Σβⱼ²]
```

- `l1_ratio = 1` → Lasso puro
- `l1_ratio = 0` → Ridge puro  
- `0 < l1_ratio < 1` → mix di entrambi

Elastic Net è particolarmente utile quando ci sono **gruppi di feature correlate**: Lasso tenderebbe a selezionarne una sola per gruppo e ignorare le altre; Elastic Net, grazie alla componente Ridge, distribuisce il peso tra le feature correlate, mantenendole tutte (o quasi) nel modello. Il prezzo da pagare è la complessità della ricerca: abbiamo due iperparametri da ottimizzare invece di uno.

In [ ]:
# ============================================================
# ELASTIC NET — RICERCA DI DUE IPERPARAMETRI: alpha e l1_ratio
# ============================================================
# La griglia è bidimensionale: per ogni combinazione (alpha, l1_ratio)
# viene eseguita una 5-fold CV. n_jobs=-1 sfrutta tutti i core disponibili.
# Limitiamo la griglia a 25 valori di alpha e 9 di l1_ratio per contenere
# i tempi di calcolo pur mantenendo una ricerca sufficientemente dettagliata.

alphas_en   = np.logspace(-3, 3, 25)
l1_ratios   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

en_gs = GridSearchCV(
    ElasticNet(max_iter=20000),
    param_grid={'alpha': alphas_en, 'l1_ratio': l1_ratios},
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1   # parallelizzazione su tutti i core
)
en_gs.fit(X_train_s, y_train)

ALPHA_EN    = en_gs.best_params_['alpha']
L1RATIO_EN  = en_gs.best_params_['l1_ratio']

print(f'Parametri ottimali per Elastic Net:')
print(f'  alpha:    {ALPHA_EN:.6f}')
print(f'  l1_ratio: {L1RATIO_EN:.2f}')
print(f'  → Composizione: {L1RATIO_EN*100:.0f}% Lasso + {(1-L1RATIO_EN)*100:.0f}% Ridge')
print(f'CV-MSE migliore: {-en_gs.best_score_:,.2f}')

In [ ]:
# ============================================================
# VISUALIZZAZIONE: CV-MSE per ogni l1_ratio al variare di alpha
# ============================================================
# Ogni linea rappresenta un valore di l1_ratio.
# Si vede come il comportamento del modello cambia al variare di
# entrambi gli iperparametri simultaneamente.

results_df = pd.DataFrame(en_gs.cv_results_)

plt.figure(figsize=(12, 5))
cmap = plt.cm.get_cmap('RdYlGn_r', len(l1_ratios))

for i, l1r in enumerate(l1_ratios):
    mask = results_df['param_l1_ratio'] == l1r
    subset = results_df[mask].sort_values('param_alpha')
    plt.plot(
        subset['param_alpha'].values,
        -subset['mean_test_score'].values,
        label=f'l1_ratio={l1r:.1f}',
        color=cmap(i),
        linewidth=1.5,
        alpha=0.85
    )

plt.axvline(x=ALPHA_EN, color='black', linestyle='--', linewidth=2,
            label=f'Alpha ottimale = {ALPHA_EN:.4f}')
plt.xscale('log')
plt.xlabel('Alpha (scala logaritmica)')
plt.ylabel('CV-MSE')
plt.title('Elastic Net: CV-MSE per ogni combinazione alpha / l1_ratio', fontsize=12)
plt.legend(fontsize=8, ncol=2, loc='upper left')
plt.tight_layout()
plt.show()

**Osservazione sul grafico Elastic Net**

Elastic Net combina due tipi di regolarizzazione: una parte simile a Ridge e una parte simile a Lasso. Per questo il grafico mostra più linee: ogni linea corrisponde a un diverso valore di `l1_ratio`. Valori bassi di `l1_ratio` rendono Elastic Net più vicino a Ridge; valori alti lo rendono più vicino a Lasso.

Nel risultato ottenuto, l'`alpha` migliore è `0.1000`, indicato dalla linea verticale tratteggiata. Questo valore si trova nella zona sinistra del grafico, dove il CV-MSE è basso e le curve sono quasi piatte. Significa che il modello beneficia di una regolarizzazione leggera, non di una penalità molto forte.

La parte destra del grafico è la più importante da interpretare: quando `alpha` aumenta troppo, tutte le curve salgono rapidamente. In pratica, la penalità diventa eccessiva, i coefficienti vengono ridotti troppo e il modello perde capacità predittiva. Questo peggioramento si vede per tutti i valori di `l1_ratio`, anche se le curve più vicine a Lasso tendono a peggiorare un po' più lentamente.

Conclusione: per questo dataset Elastic Net funziona meglio con una regolarizzazione contenuta. Il grafico non suggerisce di aumentare molto `alpha`; al contrario, mostra che oltre una certa soglia la penalità danneggia chiaramente il modello.

In [ ]:
# ============================================================
# ELASTIC NET — ADDESTRAMENTO E VALUTAZIONE FINALE
# ============================================================

en_finale = ElasticNet(alpha=ALPHA_EN, l1_ratio=L1RATIO_EN, max_iter=20000)
risultati_en = evaluate_regression_model(
    'ElasticNet', en_finale,
    X_train_s, X_test_s, y_train, y_test
)

n_non_zero_en = conta_coeff_non_nulli(en_finale)
n_zero_en     = len(FEATURE_NAMES) - n_non_zero_en

print(f'=== ELASTIC NET (alpha={ALPHA_EN:.6f}, l1_ratio={L1RATIO_EN:.2f}) ===')
print(f"MSE:             {risultati_en['MSE']:>18,.2f}")
print(f"RMSE:            {risultati_en['RMSE']:>18,.2f}")
print(f"MAE:             {risultati_en['MAE']:>18,.2f}")
print(f"R²:              {risultati_en['R²']:>18.4f}")
print(f"CV-MSE (media):  {risultati_en['CV-MSE (media)']:>18,.2f}")
print(f"CV-MSE (std):    {risultati_en['CV-MSE (std)']:>18,.2f}")
print(f'Coeff. non nulli: {n_non_zero_en}/{len(FEATURE_NAMES)}')
print(f'Coeff. azzerati:  {n_zero_en}/{len(FEATURE_NAMES)}')

**Considerazioni su Elastic Net**

Nel risultato finale Elastic Net usa `alpha = 0.100000` e `l1_ratio = 0.20`. Questo valore di `l1_ratio` è più vicino a Ridge che a Lasso: significa che il modello sta privilegiando la stabilizzazione dei coefficienti più che la selezione aggressiva delle variabili.

Infatti il risultato conferma questa lettura: `Coeff. non nulli: 12/12` e `Coeff. azzerati: 0/12`. Elastic Net, in questo caso, non elimina nessuna feature. Si comporta quindi più come un modello che distribuisce e controlla i pesi, non come un modello che semplifica davvero il numero di predittori.

Le metriche sono molto vicine a quelle di Ridge: `RMSE ≈ 1,332,561`, `MAE ≈ 958,144`, `R² ≈ 0.6487` e `CV-MSE ≈ 1,070,400,494,520`. Questo suggerisce che Elastic Net non porta un miglioramento evidente rispetto a Ridge, almeno con questo dataset.

Conclusione: Elastic Net resta un buon confronto perché è più flessibile, ma qui non sfrutta davvero la parte "Lasso" per eliminare variabili. Il suo comportamento finale è molto simile a Ridge: mantiene tutte le feature e applica una regolarizzazione leggera.

## SEZIONE 10: CONFRONTO DELLE PERFORMANCE

Raccogliamo tutti i risultati in una tabella comparativa e li visualizziamo graficamente. Ricordiamo che la scelta del modello migliore non dipende solo dalla metrica: un modello con MSE leggermente peggiore ma con molti meno coefficienti potrebbe essere preferibile in un contesto operativo dove l'interpretabilità conta.

In [ ]:
# ============================================================
# TABELLA RIEPILOGATIVA DI TUTTI I MODELLI
# ============================================================

modelli_valutati = [
    ('OLS',        lr_baseline,  risultati_ols),
    ('Ridge',      ridge_finale, risultati_ridge),
    ('Lasso',      lasso_finale, risultati_lasso),
    ('ElasticNet', en_finale,    risultati_en),
]

righe = []
for nome, modello, res in modelli_valutati:
    riga = {
        'Modello':          nome,
        'MSE':              res['MSE'],
        'RMSE':             res['RMSE'],
        'MAE':              res['MAE'],
        'R²':               res['R²'],
        'CV-MSE (media)':   res['CV-MSE (media)'],
        'CV-MSE (std)':     res['CV-MSE (std)'],
        'Coeff. non nulli': conta_coeff_non_nulli(modello)
    }
    righe.append(riga)

df_confronto = pd.DataFrame(righe).set_index('Modello')

# Formatting per leggibilità
df_display = df_confronto.copy()
for col in ['MSE', 'RMSE', 'MAE', 'CV-MSE (media)', 'CV-MSE (std)']:
    df_display[col] = df_display[col].apply(lambda x: f'{x:,.2f}')
df_display['R²'] = df_display['R²'].apply(lambda x: f'{x:.4f}')

print('=== RIEPILOGO COMPARATIVO ===\n')
print(df_display.to_string())

# Evidenziamo il vincitore per ciascuna metrica
print('\n--- Modello con MSE più basso:         ', df_confronto['MSE'].idxmin())
print('--- Modello con R² più alto:           ', df_confronto['R²'].idxmax())
print('--- Modello con MAE più basso:         ', df_confronto['MAE'].idxmin())
print('--- Modello con CV-MSE più basso:      ', df_confronto['CV-MSE (media)'].idxmin())
print('--- Modello più parsimonioso (< coeff):', df_confronto['Coeff. non nulli'].idxmin())

In [ ]:
# ============================================================
# GRAFICI COMPARATIVI: MSE, R², MAE, Coefficienti non nulli
# ============================================================

modelli_label = df_confronto.index.tolist()
colori_bar    = [COLORI[m] for m in modelli_label]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- MSE ---
bars = axes[0, 0].bar(modelli_label, df_confronto['MSE'], color=colori_bar,
                       edgecolor='white', linewidth=0.5)
axes[0, 0].set_title('Mean Squared Error (MSE)\npiù basso = meglio', fontsize=11)
axes[0, 0].set_ylabel('MSE')
for bar, val in zip(bars, df_confronto['MSE']):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() * 1.01,
                    f'{val:,.0f}', ha='center', va='bottom', fontsize=8)

# --- R² ---
bars = axes[0, 1].bar(modelli_label, df_confronto['R²'], color=colori_bar,
                       edgecolor='white', linewidth=0.5)
axes[0, 1].set_title('R² Score\npiù alto = meglio', fontsize=11)
axes[0, 1].set_ylabel('R²')
axes[0, 1].set_ylim(0, 1.05)
for bar, val in zip(bars, df_confronto['R²']):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# --- MAE ---
bars = axes[1, 0].bar(modelli_label, df_confronto['MAE'], color=colori_bar,
                       edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Mean Absolute Error (MAE)\npiù basso = meglio', fontsize=11)
axes[1, 0].set_ylabel('MAE')
for bar, val in zip(bars, df_confronto['MAE']):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() * 1.01,
                    f'{val:,.0f}', ha='center', va='bottom', fontsize=8)

# --- Coefficienti non nulli ---
bars = axes[1, 1].bar(modelli_label, df_confronto['Coeff. non nulli'], color=colori_bar,
                       edgecolor='white', linewidth=0.5)
axes[1, 1].set_title('Coefficienti non nulli\n(complessità del modello)', fontsize=11)
axes[1, 1].set_ylabel('N. coefficienti attivi')
axes[1, 1].axhline(y=len(FEATURE_NAMES), color='gray', linestyle='--',
                    alpha=0.5, label=f'Totale feature: {len(FEATURE_NAMES)}')
axes[1, 1].legend(fontsize=9)
for bar, val in zip(bars, df_confronto['Coeff. non nulli']):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.1,
                    str(int(val)), ha='center', va='bottom',
                    fontsize=10, fontweight='bold')

plt.suptitle('Confronto Performance dei Modelli', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Osservazione sui grafici di confronto**

Guardando i quattro grafici, la prima cosa che emerge è che i modelli sono molto vicini tra loro. Non c'è un modello che stacca nettamente gli altri: MSE, MAE e R² cambiano poco da una barra all'altra. Questo ci dice che, almeno su questo dataset, la regolarizzazione non trasforma radicalmente le prestazioni rispetto alla regressione lineare di base.

Lasso risulta leggermente migliore su alcune metriche: ha il MSE più basso e l'R² più alto tra i modelli confrontati. La differenza però è contenuta, quindi va letta come un piccolo vantaggio, non come una vittoria schiacciante. Ridge ed Elastic Net restano molto vicini, mentre OLS è competitivo nonostante non applichi nessuna penalità.

Il grafico in basso a destra è particolarmente utile perché mostra la complessità del modello. Tutti e quattro i modelli usano 12 coefficienti non nulli su 12. Questo significa che, in questa esecuzione, né Lasso né Elastic Net hanno eliminato variabili. Quindi il vantaggio atteso in termini di semplicità non si è davvero manifestato.

La lettura complessiva è questa: le tecniche regolarizzate aiutano a controllare meglio i coefficienti, ma non cambiano drasticamente la qualità predittiva. Se dovessi scegliere solo guardando questi grafici, Lasso sarebbe leggermente favorito per l'errore più basso; se invece cercassi stabilità e comportamento più conservativo, Ridge rimarrebbe una scelta molto solida.

In [ ]:
# ============================================================
# CROSS-VALIDATION: DISTRIBUZIONE DEL MSE SUI 5 FOLD
# ============================================================
# Il boxplot mostra non solo il CV-MSE medio, ma anche la sua variabilità.
# Un modello con media bassa ma std alta è meno affidabile di uno
# con media leggermente più alta ma std bassa — è più stabile.

cv_data   = []
cv_labels = []

for nome, modello, _ in modelli_valutati:
    cv_scores = cross_val_score(
        modello, X_train_s, y_train,
        cv=5, scoring='neg_mean_squared_error'
    )
    cv_data.append(-cv_scores)
    cv_labels.append(nome)

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(cv_data, labels=cv_labels, patch_artist=True, notch=False)

for patch, nome in zip(bp['boxes'], cv_labels):
    patch.set_facecolor(COLORI[nome])
    patch.set_alpha(0.75)

for element in ['whiskers', 'caps', 'medians', 'fliers']:
    for item in bp[element]:
        item.set(color='#333333', linewidth=1.2)

ax.set_ylabel('MSE (5-fold CV)')
ax.set_title('Distribuzione del MSE in Cross-Validation (5 fold)\n'
             'La linea arancione è la mediana', fontsize=12)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('Riepilogo CV per fold:')
print(f'{"Modello":<15} {"Media":>12} {"Std":>12} {"Min":>12} {"Max":>12}')
print('-' * 65)
for nome, data in zip(cv_labels, cv_data):
    print(f'{nome:<15} {data.mean():>12,.2f} {data.std():>12,.2f} '
          f'{data.min():>12,.2f} {data.max():>12,.2f}')

**Osservazione sul boxplot della cross-validation**

Questo grafico mostra come si comportano i modelli nei 5 fold della cross-validation. Non stiamo quindi guardando un solo errore finale, ma cinque errori diversi per ogni modello, ottenuti cambiando ogni volta la porzione usata come validazione. È un modo più onesto per capire se il modello è stabile oppure se funziona bene solo con una certa divisione dei dati.

A colpo d'occhio i quattro box sono molto simili: hanno mediane vicine e un'ampiezza comparabile. Anche la tabella sotto il grafico conferma questa lettura: le medie del CV-MSE sono tutte intorno a `1.07e12` e le deviazioni standard sono molto vicine tra loro. Quindi nessun modello mostra una stabilità nettamente superiore agli altri.

Ridge ed Elastic Net hanno la media CV-MSE leggermente più bassa, mentre Lasso e OLS sono appena sopra. La differenza però è piccola rispetto alla variabilità tra i fold: questo significa che non conviene esagerare con le conclusioni. Più che decretare un vincitore assoluto, il grafico suggerisce che tutti i modelli hanno una capacità di generalizzazione abbastanza simile.

**Lettura del confronto**

Mettendo insieme grafici e tabella, il quadro è abbastanza equilibrato. Sul test set Lasso sembra leggermente più avanti, perché ottiene il MSE più basso e l'R² più alto. In cross-validation, però, Ridge ed Elastic Net risultano appena migliori come media del CV-MSE. Questa differenza è utile da notare: il test set racconta una singola divisione dei dati, mentre la cross-validation guarda più suddivisioni e quindi pesa di più la stabilità.

La cosa più importante è che le distanze tra i modelli sono piccole. Non siamo davanti a un caso in cui un algoritmo batte chiaramente tutti gli altri. I modelli stanno lavorando più o meno nella stessa fascia di errore, segno che il limite probabilmente non è tanto nella scelta tra Ridge, Lasso ed Elastic Net, ma nelle informazioni disponibili nel dataset.

Dal punto di vista pratico, sceglierei Ridge o Elastic Net se l'obiettivo principale fosse avere un modello regolare e stabile. Sceglierei Lasso solo se volessi privilegiare il risultato sul test set, ma in questa esecuzione non offre il vantaggio tipico della selezione delle feature, perché mantiene comunque 12 coefficienti su 12.

## SEZIONE 11: ANALISI DEI COEFFICIENTI E PERCORSI DI REGOLARIZZAZIONE

L'analisi dei coefficienti è forse la parte più informativa dal punto di vista interpretativo. Confrontando i coefficienti dei quattro modelli vediamo direttamente come la regolarizzazione agisce: Ridge li comprime, Lasso ne azzera alcuni, Elastic Net fa entrambe le cose in misura diversa.

I **percorsi di regolarizzazione** mostrano come i coefficienti evolvono al variare di alpha — dalla soluzione OLS (alpha→0) fino a quasi zero (alpha→∞). Sono grafici molto usati in pratica per capire quali feature "resistono" alla regolarizzazione (quelle davvero predittive) e quali vengono eliminate per prime.

In [ ]:
# ============================================================
# CONFRONTO TABELLARE DEI COEFFICIENTI
# ============================================================

df_coefficienti = pd.DataFrame({
    'OLS':        lr_baseline.coef_,
    'Ridge':      ridge_finale.coef_,
    'Lasso':      lasso_finale.coef_,
    'ElasticNet': en_finale.coef_
}, index=FEATURE_NAMES)

# Ordiniamo per importanza assoluta nel modello OLS (come riferimento)
df_coefficienti = df_coefficienti.reindex(
    df_coefficienti['OLS'].abs().sort_values(ascending=False).index
)

print('=== COEFFICIENTI PER MODELLO (dati scalati) ===')
print(df_coefficienti.round(2).to_string())

print('\nLegenda: coefficienti a 0.00 sono stati azzerati dalla regolarizzazione.')

In [ ]:
# ============================================================
# GRAFICO: CONFRONTO VISIVO DEI COEFFICIENTI
# ============================================================

x       = np.arange(len(FEATURE_NAMES))
width   = 0.2
ordine  = df_coefficienti.index.tolist()

fig, ax = plt.subplots(figsize=(15, 6))

for i, (col_nome, colore) in enumerate([
    ('OLS',        COLORI['OLS']),
    ('Ridge',      COLORI['Ridge']),
    ('Lasso',      COLORI['Lasso']),
    ('ElasticNet', COLORI['ElasticNet'])
]):
    ax.bar(x + i*width,
           df_coefficienti.loc[ordine, col_nome],
           width, label=col_nome, color=colore, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(ordine, rotation=25, ha='right', fontsize=9)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('Valore del coefficiente (dati scalati)')
ax.set_title('Confronto dei Coefficienti per Modello', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Osservazione sul grafico dei coefficienti**

Questo grafico mostra il peso che ogni modello assegna alle variabili dopo la standardizzazione. Proprio perché le feature sono state scalate, le barre sono confrontabili tra loro: una barra più alta indica che quella variabile sta influenzando di più la previsione del prezzo rispetto alle altre.

La prima cosa che si nota è che `bathrooms` e `area` hanno i coefficienti più alti in tutti i modelli. È un risultato molto plausibile: il numero di bagni e la superficie sono tra gli elementi più forti nella valutazione di una casa. Subito dopo troviamo variabili come `airconditioning`, `stories` e `prefarea`, che descrivono comfort, struttura e posizione dell'immobile.

Le barre dei quattro modelli sono abbastanza simili. Questo significa che Ridge, Lasso ed Elastic Net non stanno cambiando completamente il modo in cui il modello interpreta i dati rispetto a OLS. La regolarizzazione modifica un po' l'intensità dei coefficienti, ma non ribalta l'importanza delle variabili principali.

Un dettaglio importante è che Lasso non azzera nessun coefficiente: tutte le variabili restano presenti nel grafico. Quindi, in questo caso, Lasso non sta semplificando il modello eliminando feature; sta solo ridistribuendo leggermente i pesi. Le variabili con peso più basso, come `bedrooms`, `guestroom` e `furnishingstatus`, restano comunque nel modello, ma incidono meno rispetto a `bathrooms` e `area`.

In [ ]:
# ============================================================
# PERCORSI DI REGOLARIZZAZIONE: RIDGE e LASSO
# ============================================================
# Per ogni valore di alpha, addestriamo il modello e salviamo i coefficienti.
# Il grafico risultante mostra come ogni feature reagisce all'aumento di alpha:
# - In Ridge: tutti convergono a zero gradualmente (mai raggiungono lo zero)
# - In Lasso: alcuni raggiungono zero prima di altri (feature selection)

alphas_path = np.logspace(-2, 6, 150)

# ---- RIDGE PATH ----
coeff_ridge_path = []
for a in alphas_path:
    r_tmp = Ridge(alpha=a, max_iter=10000)
    r_tmp.fit(X_train_s, y_train)
    coeff_ridge_path.append(r_tmp.coef_)
coeff_ridge_path = np.array(coeff_ridge_path)

# ---- LASSO PATH ----
# Lasso azzera tutto prima di alpha=1e4, inutile andare oltre
alphas_lasso_path = np.logspace(-4, 4, 150)
coeff_lasso_path = []
for a in alphas_lasso_path:
    l_tmp = Lasso(alpha=a, max_iter=20000)
    l_tmp.fit(X_train_s, y_train)
    coeff_lasso_path.append(l_tmp.coef_)
    if np.all(np.abs(l_tmp.coef_) < 1e-10):  # tutti zero → usciamo prima
        alphas_lasso_path = alphas_lasso_path[:len(coeff_lasso_path)]
        break
coeff_lasso_path = np.array(coeff_lasso_path)

# ---- VISUALIZZAZIONE ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap_path = plt.cm.get_cmap('tab10', len(FEATURE_NAMES))

for j, feat in enumerate(FEATURE_NAMES):
    axes[0].plot(alphas_path, coeff_ridge_path[:, j],
                 label=feat, color=cmap_path(j), linewidth=1.8)
axes[0].axvline(x=ALPHA_RIDGE, color='black', linestyle='--', linewidth=1.5,
                label=f'α ottimale={ALPHA_RIDGE:.2f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (scala log)')
axes[0].set_ylabel('Valore coefficiente')
axes[0].set_title('Percorso di regolarizzazione — Ridge', fontsize=12)
axes[0].legend(fontsize=7, ncol=2, loc='upper right')
axes[0].grid(alpha=0.3)

for j, feat in enumerate(FEATURE_NAMES):
    axes[1].plot(alphas_lasso_path, coeff_lasso_path[:, j],
                 label=feat, color=cmap_path(j), linewidth=1.8)
axes[1].axvline(x=ALPHA_LASSO, color='black', linestyle='--', linewidth=1.5,
                label=f'α ottimale={ALPHA_LASSO:.4f}')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (scala log)')
axes[1].set_ylabel('Valore coefficiente')
axes[1].set_title('Percorso di regolarizzazione — Lasso', fontsize=12)
axes[1].legend(fontsize=7, ncol=2, loc='upper right')
axes[1].grid(alpha=0.3)

plt.suptitle('Percorsi di Regolarizzazione: evoluzione dei coefficienti con alpha',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Cosa raccontano i percorsi di regolarizzazione**

Questi due grafici fanno vedere una cosa che dalle sole metriche non si capisce: come cambiano i pesi delle variabili quando aumentiamo `alpha`. Ogni linea è una feature del dataset. Se una linea è alta, quella variabile ha un peso importante nella previsione; se scende verso zero, il modello le sta dando sempre meno importanza.

Nel grafico di **Ridge** si vede bene il suo comportamento tipico. Per valori piccoli di `alpha` le linee restano quasi ferme: il modello è molto simile alla regressione lineare classica. Quando `alpha` cresce, soprattutto da circa `10²` in poi, le curve iniziano a scendere tutte insieme verso zero. Ridge quindi non sceglie una variabile da eliminare, ma riduce gradualmente il peso di tutte. È come se rendesse il modello più prudente, senza fare tagli netti.

Nel grafico di **Lasso**, invece, in questo specifico esperimento le linee rimangono quasi piatte fino al valore ottimale scelto (`alpha ≈ 2616.50`). Questo è coerente con il risultato visto prima: Lasso non ha azzerato nessun coefficiente, infatti tutti i 12 predittori restano attivi. Quindi qui Lasso non sta facendo una vera selezione delle feature; sta solo riducendo leggermente alcuni pesi.

La differenza più interessante è questa: Ridge mostra chiaramente l'effetto della penalità quando `alpha` diventa grande, mentre Lasso, nel range utile per questo dataset, mantiene ancora tutte le variabili. Per eliminare feature servirebbe una penalità più forte, ma dai grafici precedenti sappiamo che aumentare troppo `alpha` fa peggiorare il CV-MSE.

La linea verticale tratteggiata indica il valore di `alpha` scelto dalla cross-validation. In entrambi i casi è il punto in cui il notebook prova a bilanciare due esigenze: non lasciare i coefficienti completamente liberi, ma nemmeno ridurli così tanto da perdere informazioni utili.

## SEZIONE 12: ANALISI DEI RESIDUI

I residui sono la differenza tra il valore reale e quello previsto dal modello: `residuo = y_reale - y_predetto`. Analizzarli è fondamentale per valutare se il modello è ben specificato. In una regressione lineare correttamente impostata ci aspettiamo residui:

1. **Con media zero**: il modello non è sistematicamente ottimistico o pessimistico
2. **Normalmente distribuiti**: verifica che l'assunzione di normalità degli errori regga
3. **Omoscedastici**: la varianza degli errori non deve dipendere dai valori predetti
4. **Senza pattern**: se si vedono strutture nei residui, il modello sta perdendo qualcosa

Analizzare i residui di tutti e quattro i modelli ci permette di confrontare non solo le performance numeriche, ma anche la qualità dell'adattamento.

In [ ]:
# ============================================================
# ISTOGRAMMA DEI RESIDUI + RESIDUI VS VALORI PREDETTI
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col_idx, (nome, modello, res) in enumerate(modelli_valutati):
    y_pred = res['y_pred']
    residui = y_test.values - y_pred
    colore  = COLORI[nome]

    # Riga 1: Distribuzione dei residui
    axes[0, col_idx].hist(residui, bins=25, color=colore,
                           alpha=0.8, edgecolor='white', density=False)
    axes[0, col_idx].axvline(x=0, color='black', linewidth=1.5, label='Media ideale=0')
    axes[0, col_idx].axvline(x=np.mean(residui), color='red', linewidth=1.2,
                              linestyle='--', label=f'Media={np.mean(residui):,.0f}')
    axes[0, col_idx].set_title(f'{nome}\nDistribuzione residui', fontsize=10)
    axes[0, col_idx].set_xlabel('Residuo')
    axes[0, col_idx].set_ylabel('Frequenza')
    axes[0, col_idx].legend(fontsize=7)

    # Riga 2: Residui vs Valori Predetti
    # Cerchiamo pattern: se c'è una curva o un funnel, il modello ha problemi
    axes[1, col_idx].scatter(y_pred, residui, alpha=0.45, color=colore, s=18)
    axes[1, col_idx].axhline(y=0, color='black', linewidth=1.2, linestyle='--')
    # Linea di tendenza per evidenziare eventuali pattern sistematici
    z = np.polyfit(y_pred, residui, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(y_pred.min(), y_pred.max(), 100)
    axes[1, col_idx].plot(x_trend, p(x_trend), color='red',
                           linewidth=1.5, label='Tendenza')
    axes[1, col_idx].set_title(f'{nome}\nResidui (errori) vs Predetti', fontsize=10)
    axes[1, col_idx].set_xlabel('Valori predetti')
    axes[1, col_idx].set_ylabel('Residuo')
    axes[1, col_idx].legend(fontsize=7)

plt.suptitle('Analisi dei Residui per ogni Modello', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Osservazione sui residui**

Questo blocco confronta due aspetti degli errori commessi dai modelli. Nella riga in alto vediamo la distribuzione dei residui: idealmente dovrebbe essere centrata intorno a zero, perché vorrebbe dire che il modello non tende sempre a sovrastimare o sempre a sottostimare. La linea nera indica lo zero ideale, mentre la linea rossa tratteggiata mostra la media reale dei residui.

Nei quattro modelli la distribuzione è abbastanza simile: la maggior parte degli errori resta concentrata vicino allo zero, ma ci sono anche alcuni residui molto grandi. Questo è coerente con quanto visto prima sui prezzi: gli immobili più costosi o particolari sono più difficili da stimare bene.

La riga in basso mostra i residui rispetto ai valori predetti. Qui non cerchiamo una linea perfetta, ma l'assenza di pattern troppo evidenti. Se i punti fossero distribuiti in modo casuale intorno allo zero, sarebbe il caso ideale. Invece si nota una leggera tendenza crescente: per valori predetti più alti, gli errori tendono ad aumentare. Questo suggerisce che il modello fatica soprattutto nella fascia alta del mercato, dove probabilmente mancano variabili più specifiche come posizione precisa, qualità dell'immobile o anno di costruzione.

In [ ]:
# ============================================================
# Q-Q PLOT: VERIFICA DELLA NORMALITÀ DEI RESIDUI
# ============================================================
# Il Q-Q plot confronta i quantili empirici dei residui con quelli
# attesi da una distribuzione normale teorica.
# Se i punti seguono la retta rossa → residui normalmente distribuiti.
# Deviazioni alle code indicano code più pesanti del normale (outlier).

# Test di Shapiro-Wilk: test statistico formale di normalità.
# H0: i residui provengono da una distribuzione normale.
# p > 0.05 → non rifiutiamo H0 (compatibili con la normalità)
# p < 0.05 → rifiutiamo H0 (deviazioni significative dalla normalità)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for col_idx, (nome, modello, res) in enumerate(modelli_valutati):
    y_pred  = res['y_pred']
    residui = y_test.values - y_pred
    colore  = COLORI[nome]

    (osm, osr), (slope, intercept, r_sq) = stats.probplot(residui, dist='norm')

    axes[col_idx].scatter(osm, osr, color=colore, s=15, alpha=0.7, label='Residui')
    # Retta di riferimento normale
    x_line = np.array([osm.min(), osm.max()])
    axes[col_idx].plot(x_line, slope * x_line + intercept,
                        color='red', linewidth=1.5, label='Distribuzione normale')

    stat_sw, p_sw = stats.shapiro(residui)
    axes[col_idx].set_title(f'{nome}\nShapiro p={p_sw:.3f}', fontsize=10)
    axes[col_idx].set_xlabel('Quantili teorici')
    axes[col_idx].set_ylabel('Quantili empirici')
    axes[col_idx].legend(fontsize=7)

plt.suptitle('Q-Q Plot dei Residui (verifica normalità)', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.show()

**Osservazione sul Q-Q plot**

Il Q-Q plot serve a controllare se i residui assomigliano a una distribuzione normale. Se i punti seguono bene la linea rossa, gli errori hanno una forma abbastanza vicina alla normalità. Quando invece i punti si allontanano dalla linea, soprattutto alle estremità, significa che gli errori più grandi sono meno regolari.

In questi grafici la parte centrale segue abbastanza bene la linea, mentre le code si allontanano. Questo vuol dire che il modello gestisce discretamente la maggior parte degli immobili, ma fa più fatica sui casi estremi, cioè quelli con errori molto alti in positivo o in negativo.

In [ ]:
# ============================================================
# VALORI REALI vs VALORI PREDETTI
# ============================================================
# Se il modello fosse perfetto, tutti i punti cadrebbero sulla retta
# tratteggiata rossa (y_predetto = y_reale).
# La dispersione intorno a questa retta mostra l'entità degli errori.
# Pattern sistematici (es. il modello sottostima sempre i prezzi alti)
# indicano che manca qualcosa — una feature importante, una non-linearità.

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for idx, (nome, modello, res) in enumerate(modelli_valutati):
    y_pred = res['y_pred']
    colore = COLORI[nome]

    axes[idx].scatter(y_test, y_pred, alpha=0.5, color=colore, s=25, label='Campioni')

    # Linea di predizione perfetta
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[idx].plot([min_val, max_val], [min_val, max_val],
                   'r--', linewidth=1.8, label='Pred. perfetta')

    axes[idx].set_xlabel('Valori reali (price)')
    axes[idx].set_ylabel('Valori predetti')
    r2_val = res['R²']
    rmse_val = res['RMSE']
    axes[idx].set_title(
        f'{nome}  (R²={r2_val:.4f}, RMSE={rmse_val:,.0f})', fontsize=11
    )
    axes[idx].legend(fontsize=9)

plt.suptitle('Valori Reali vs Valori Predetti', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Osservazione sui valori reali vs predetti**

In questi grafici ogni punto rappresenta un immobile: sull'asse orizzontale c'è il prezzo reale, su quello verticale il prezzo previsto dal modello. La linea rossa tratteggiata rappresenta la previsione perfetta: se un punto cade esattamente su quella linea, significa che il modello ha previsto il prezzo corretto.

I punti seguono abbastanza bene la diagonale nella fascia centrale dei prezzi, quindi i modelli riescono a catturare la tendenza generale: immobili con prezzo reale più alto tendono ad avere anche una previsione più alta. Questo è coerente con gli R² intorno a 0.65: il modello spiega una buona parte del fenomeno, ma non tutto.

La parte più problematica è la fascia alta: diversi immobili costosi vengono previsti sotto la linea rossa, quindi il modello tende a sottostimare alcune case di valore elevato. È un comportamento abbastanza comune nei modelli lineari su dati immobiliari, perché le case più costose spesso dipendono da dettagli che nel dataset non sono presenti, come zona precisa, qualità delle finiture, vista, anno di costruzione o stato dell'immobile.

Anche qui i quattro modelli si comportano in modo molto simile. Lasso ha un R² leggermente più alto e un RMSE leggermente più basso, ma la forma complessiva degli errori resta quasi la stessa.

In [ ]:
# ============================================================
# STATISTICHE RIASSUNTIVE DEI RESIDUI
# ============================================================

print(f'{'Modello':<15} {'Media':>12} {'Std':>12} {'Min':>14} {'Max':>14}')
print('-' * 70)
for nome, modello, res in modelli_valutati:
    residui = y_test.values - res['y_pred']
    print(f'{nome:<15} {np.mean(residui):>12,.2f} {np.std(residui):>12,.2f} '
          f'{np.min(residui):>14,.2f} {np.max(residui):>14,.2f}')

print('\nNota: la media dei residui dovrebbe essere vicina a zero per tutti i modelli.')
print('Una media negativa indica che il modello tende a sovrastimare il prezzo,')
print('una media positiva che tende a sottostimare.')

**Lettura dell'analisi dei residui**

I residui indicano quanto il modello sbaglia per ogni immobile: se il residuo è vicino a zero, la previsione è vicina al prezzo reale; se è molto positivo o molto negativo, il modello ha sottostimato o sovrastimato in modo più marcato.

Gli istogrammi dei residui sono abbastanza simili tra i modelli. Questo suggerisce che Ridge, Lasso ed Elastic Net non cambiano completamente il tipo di errore commesso: intervengono soprattutto sulla stabilità dei coefficienti e sulla capacità di generalizzare, più che sulla forma generale degli errori.

Nel grafico "Residui vs Valori Predetti" bisogna cercare eventuali forme strane: curve, ventagli o gruppi separati. Se la dispersione aumenta sui prezzi più alti, significa che il modello fatica di più sulle case costose. È abbastanza normale: gli immobili di fascia alta sono pochi e spesso hanno caratteristiche particolari che non sempre sono descritte bene dalle feature disponibili.

Il Q-Q plot serve a controllare se i residui assomigliano a una distribuzione normale. Se i punti si allontanano dalla linea soprattutto alle estremità, vuol dire che gli errori più grandi sono meno regolari. Non è automaticamente un problema grave, ma è un segnale utile: il modello lineare funziona discretamente, però non cattura tutta la complessità del mercato immobiliare.

## CONCLUSIONI FINALI

### Cosa emerge dal confronto

Il confronto finale mostra che tutti i modelli lavorano in una fascia di performance molto simile. Non emerge un vincitore netto: Lasso ottiene risultati leggermente migliori sul test set, mentre Ridge ed Elastic Net risultano appena più favorevoli guardando la media del CV-MSE in cross-validation. Le differenze, però, sono contenute.

Questo suggerisce che il dataset non soffre di un overfitting grave risolvibile semplicemente cambiando tecnica di regolarizzazione. I modelli regolarizzati aiutano soprattutto a controllare i coefficienti e a rendere il modello più stabile, ma non modificano drasticamente la capacità predittiva rispetto alla regressione lineare classica.

**Ridge** mantiene tutte le feature nel modello (`12/12`) e applica una regolarizzazione moderata (`alpha ≈ 29.15`). È la scelta più conservativa: non semplifica il numero di variabili, ma rende i coefficienti più controllati e il modello più regolare.

**Lasso**, dopo il test con valori di `alpha` più alti, usa `alpha ≈ 2616.50`. Sul test set è leggermente il migliore tra i modelli confrontati, ma non azzera nessun coefficiente (`0/12`). Quindi in questo caso non produce un modello più snello: mantiene tutte le variabili e si limita a modificarne i pesi.

**Elastic Net** usa `alpha = 0.10` e `l1_ratio = 0.20`, quindi si comporta più vicino a Ridge che a Lasso. Anche lui mantiene tutte le feature (`12/12`) e non porta un vantaggio evidente rispetto a Ridge, pur restando utile come modello di confronto perché combina entrambe le penalità.

In sintesi: se l'obiettivo è massimizzare leggermente la performance sul test set, Lasso è il candidato più interessante. Se invece si preferisce un modello più stabile e conservativo, Ridge rimane una scelta molto solida. Elastic Net non aggiunge molto in questo specifico esperimento.